# All Results — HetLoRA-M
Loads from all result directories. Includes SPA-M. Deduplicates by (method, dataset, alpha, seed).

In [ ]:
import os, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Path config ──────────────────────────────────────────────────────────────
REWORK = os.path.expanduser('~/FedLLM-Re/rework')

# All directories to scan per dataset — ordered by PREFERENCE (later = higher priority for dedup)
# Earlier dirs = older/partial results; later dirs = newer/more complete
SCAN_DIRS = {
    'yelp': [
        os.path.join(REWORK, 'results', 'yelp'),
        os.path.join(REWORK, 'results', 'V2'),
        os.path.join(REWORK, 'results_v2', 'yelp'),
        os.path.join(REWORK, 'results_hetloram_b05', 'yelp'),
        os.path.join(REWORK, 'results_prof', 'yelp'),
    ],
    'gsm8k': [
        os.path.join(REWORK, 'results', 'V2', 'gsmk'),
        os.path.join(REWORK, 'results_v2', 'gsm8k'),
        os.path.join(REWORK, 'results_hetloram_b05', 'gsm8k'),
        os.path.join(REWORK, 'results_prof', 'gsm8k'),
    ],
    'alpaca': [
        os.path.join(REWORK, 'results', 'alpaca'),
        os.path.join(REWORK, 'results', 'V2', 'alpaca'),
        os.path.join(REWORK, 'results_v2', 'alpaca'),
        os.path.join(REWORK, 'results_hetloram_b05', 'alpaca'),
        os.path.join(REWORK, 'results_prof', 'alpaca'),
    ],
}

# Metric key per dataset
METRIC = {'yelp': 'accuracy', 'gsm8k': 'exact_match', 'alpaca': 'rouge_l'}

# Methods to report (in display order)
METHODS_ORDER = ['homo_r8', 'hetero_pad', 'flexlora', 'hetlora', 'spa_m', 'hetlora_m']
METHOD_LABELS = {
    'homo_r8':   'Homo r=8',
    'hetero_pad':'Hetero-Pad',
    'flexlora':  'FlexLoRA',
    'hetlora':   'HetLoRA',
    'spa_m':     'SPA-M',
    'hetlora_m': 'HetLoRA-M (Ours)',
}

# Valid seeds only (exclude test seeds 0-4)
VALID_SEEDS = {42, 43, 44, 45, 46}

print('Config loaded. REWORK =', REWORK)

In [ ]:
# ── Load all results ──────────────────────────────────────────────────────────
def load_all(dataset):
    """
    Scan all directories for a dataset. Deduplicate by (method, alpha, seed),
    keeping the version from the latest-priority directory (highest index in SCAN_DIRS).
    Returns dict: (method, alpha, seed) -> rounds list
    """
    records = {}  # (method, alpha, seed) -> (priority, rounds)
    dirs = SCAN_DIRS[dataset]

    for priority, d in enumerate(dirs):
        if not os.path.isdir(d):
            continue
        for fpath in glob.glob(os.path.join(d, '*.json')):
            if '(1)' in fpath:  # skip duplicate files
                continue
            try:
                data = json.load(open(fpath))
            except Exception:
                continue
            method = data.get('method', '')
            seed   = int(data.get('seed', -1))
            alpha  = float(data.get('alpha', -1))
            rounds = data.get('rounds', [])
            if method not in METHODS_ORDER:
                continue
            if seed not in VALID_SEEDS:
                continue
            if not rounds:
                continue
            key = (method, alpha, seed)
            if key not in records or priority >= records[key][0]:
                records[key] = (priority, rounds)

    return {k: v[1] for k, v in records.items()}

ALL = {ds: load_all(ds) for ds in ['yelp', 'gsm8k', 'alpaca']}

# Summary
for ds, recs in ALL.items():
    print(f'\n{ds.upper()}:')
    for method in METHODS_ORDER:
        entries = {(a, s): r for (m, a, s), r in recs.items() if m == method}
        by_alpha = {}
        for (a, s), r in entries.items():
            by_alpha.setdefault(a, []).append((s, len(r)))
        if not by_alpha:
            print(f'  {METHOD_LABELS.get(method, method):20s}  — no data')
        else:
            for a in sorted(by_alpha):
                seeds_info = sorted(by_alpha[a])
                seeds_str = ', '.join(f's{s}({nr}r)' for s, nr in seeds_info)
                print(f'  {METHOD_LABELS.get(method, method):20s}  α={a}  [{seeds_str}]')

In [ ]:
# ── Compute AUC / MeanL5 / Best per run ──────────────────────────────────────
def compute_stats(rounds, metric_key, last_n=5):
    """Returns (auc, mean_last_n, best) for a single run."""
    vals = [r[metric_key] for r in rounds if metric_key in r]
    if not vals:
        return None, None, None
    scale = 100.0 if metric_key in ('accuracy', 'exact_match') else 1.0
    vals = [v * scale for v in vals]
    auc  = float(np.mean(vals))
    ml5  = float(np.mean(vals[-last_n:]))
    best = float(np.max(vals))
    return auc, ml5, best

rows = []
for ds, recs in ALL.items():
    mkey = METRIC[ds]
    for (method, alpha, seed), rounds in recs.items():
        auc, ml5, best = compute_stats(rounds, mkey)
        if auc is None:
            continue
        rows.append(dict(dataset=ds, method=method, alpha=alpha,
                         seed=seed, n_rounds=len(rounds),
                         auc=auc, mean_l5=ml5, best=best))

df = pd.DataFrame(rows)
print(f'Total runs loaded: {len(df)}')
print(df.groupby(['dataset','method'])['seed'].count().unstack('dataset').fillna(0).astype(int))

In [ ]:
# ── Main summary table: AUC ± std / MeanL5 ± std ────────────────────────────
def summary_table(dataset, alphas=None):
    sub = df[df.dataset == dataset].copy()
    if alphas is not None:
        sub = sub[sub.alpha.isin(alphas)]

    all_alphas = sorted(sub.alpha.unique())
    cols = []
    for a in all_alphas:
        cols += [f'AUC α={a}', f'MeanL5 α={a}', f'n α={a}']

    result = []
    for method in METHODS_ORDER:
        row = {'Method': METHOD_LABELS.get(method, method)}
        for a in all_alphas:
            g = sub[(sub.method == method) & (sub.alpha == a)]
            if len(g) == 0:
                row[f'AUC α={a}'] = '—'
                row[f'MeanL5 α={a}'] = '—'
                row[f'n α={a}'] = 0
            else:
                fmt = '{:.1f}±{:.1f}'.format
                row[f'AUC α={a}']    = fmt(g.auc.mean(), g.auc.std(ddof=1) if len(g)>1 else 0)
                row[f'MeanL5 α={a}'] = fmt(g.mean_l5.mean(), g.mean_l5.std(ddof=1) if len(g)>1 else 0)
                row[f'n α={a}'] = len(g)
        result.append(row)

    tbl = pd.DataFrame(result).set_index('Method')
    return tbl[cols]

print('=== YELP ===')
display(summary_table('yelp'))

print('\n=== GSM8K ===')
display(summary_table('gsm8k'))

print('\n=== ALPACA ===')
display(summary_table('alpaca'))

In [ ]:
# ── Progress tracker: how many seeds are done per method/alpha ────────────────
TARGET_SEEDS = 5
datasets_to_check = [
    ('yelp',  [0.5, 0.1, 0.01]),
    ('gsm8k', [0.5, 0.1, 0.01]),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (ds, alphas) in zip(axes, datasets_to_check):
    sub = df[df.dataset == ds]
    methods = [m for m in METHODS_ORDER if m in sub.method.values]
    x = np.arange(len(methods))
    width = 0.8 / len(alphas)
    colors = ['#2196F3', '#FF9800', '#F44336']
    for i, a in enumerate(alphas):
        counts = [len(sub[(sub.method==m) & (sub.alpha==a)]) for m in methods]
        bars = ax.bar(x + i*width - 0.4 + width/2, counts, width,
                      label=f'α={a}', color=colors[i], alpha=0.8)
        for bar, c in zip(bars, counts):
            if c > 0:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                        str(c), ha='center', va='bottom', fontsize=8)
    ax.axhline(TARGET_SEEDS, color='k', linestyle='--', linewidth=0.8, label=f'target={TARGET_SEEDS}')
    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_LABELS.get(m,m) for m in methods], rotation=30, ha='right', fontsize=8)
    ax.set_ylim(0, TARGET_SEEDS + 1)
    ax.set_title(ds.upper())
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('progress_tracker.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Convergence curves: Yelp ──────────────────────────────────────────────────
COLORS = {
    'homo_r8':   '#9E9E9E',
    'hetero_pad':'#FF9800',
    'flexlora':  '#2196F3',
    'hetlora':   '#4CAF50',
    'spa_m':     '#F44336',
    'hetlora_m': '#9C27B0',
}

def convergence_plot(dataset, alphas, metric_key, ylabel, ax_titles=None):
    recs = ALL[dataset]
    n = len(alphas)
    fig, axes = plt.subplots(1, n, figsize=(6*n, 4), sharey=False)
    if n == 1: axes = [axes]
    scale = 100.0 if metric_key in ('accuracy','exact_match') else 1.0

    for ax, alpha in zip(axes, alphas):
        for method in METHODS_ORDER:
            runs = [v for (m,a,s),v in recs.items() if m==method and a==alpha]
            if not runs: continue
            max_r = max(len(r) for r in runs)
            mat = np.full((len(runs), max_r), np.nan)
            for i, r in enumerate(runs):
                vals = [x.get(metric_key, np.nan)*scale for x in r]
                mat[i, :len(vals)] = vals
            mean = np.nanmean(mat, axis=0)
            std  = np.nanstd(mat, axis=0)
            x = np.arange(1, max_r+1)
            lw = 2.0 if method == 'hetlora_m' else 1.2
            ax.plot(x, mean, label=METHOD_LABELS.get(method,method),
                    color=COLORS.get(method,'k'), linewidth=lw)
            ax.fill_between(x, mean-std, mean+std,
                            color=COLORS.get(method,'k'), alpha=0.1)
        ax.set_title(ax_titles[alphas.index(alpha)] if ax_titles else f'α={alpha}')
        ax.set_xlabel('Round')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

fig = convergence_plot('yelp', [0.5, 0.1, 0.01], 'accuracy', 'Accuracy (%)',
                       ax_titles=['Yelp α=0.5 (moderate)', 'Yelp α=0.1 (hard)', 'Yelp α=0.01 (extreme, 40r)'])
plt.savefig('yelp_convergence_all.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Convergence curves: GSM8K ─────────────────────────────────────────────────
fig = convergence_plot('gsm8k', [0.5, 0.1, 0.01], 'exact_match', 'Exact Match (%)',
                       ax_titles=['GSM8K α=0.5', 'GSM8K α=0.1', 'GSM8K α=0.01 (40r)'])
plt.savefig('gsm8k_convergence_all.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Stability table: std of last-5 rounds (within-run oscillation) ────────────
def stability_table(dataset, alphas, metric_key):
    recs = ALL[dataset]
    scale = 100.0 if metric_key in ('accuracy','exact_match') else 1.0
    rows_out = []
    for method in METHODS_ORDER:
        row = {'Method': METHOD_LABELS.get(method, method)}
        for a in alphas:
            runs = [v for (m,al,s),v in recs.items() if m==method and al==a]
            if not runs:
                row[f'Stab α={a}'] = '—'
                continue
            stds = [np.std([x.get(metric_key,np.nan)*scale for x in r[-5:]])
                    for r in runs]
            row[f'Stab α={a}'] = f'{np.mean(stds):.2f}'
        rows_out.append(row)
    return pd.DataFrame(rows_out).set_index('Method')

print('=== YELP STABILITY (std of last-5 rounds, lower=better) ===')
display(stability_table('yelp', [0.5, 0.1, 0.01], 'accuracy'))

print('\n=== GSM8K STABILITY ===')
display(stability_table('gsm8k', [0.5, 0.1, 0.01], 'exact_match'))

In [ ]:
# ── Extended table (Yelp α=0.1): AUC / MeanL5 / Best / Best-AUC gap ──────────
def extended_table(dataset, alpha, metric_key):
    recs = ALL[dataset]
    scale = 100.0 if metric_key in ('accuracy','exact_match') else 1.0
    rows_out = []
    for method in METHODS_ORDER:
        runs = [v for (m,a,s),v in recs.items() if m==method and a==alpha]
        if not runs:
            rows_out.append({'Method': METHOD_LABELS.get(method,method),
                             'AUC':'—','MeanL5':'—','Best':'—','Best−AUC':'—','n':0})
            continue
        aucs, ml5s, bests = [], [], []
        for r in runs:
            vals = [x.get(metric_key,np.nan)*scale for x in r]
            aucs.append(np.nanmean(vals))
            ml5s.append(np.nanmean(vals[-5:]))
            bests.append(np.nanmax(vals))
        rows_out.append({
            'Method': METHOD_LABELS.get(method, method),
            'AUC':    f'{np.mean(aucs):.1f}±{np.std(aucs, ddof=1) if len(aucs)>1 else 0:.1f}',
            'MeanL5': f'{np.mean(ml5s):.1f}±{np.std(ml5s, ddof=1) if len(ml5s)>1 else 0:.1f}',
            'Best':   f'{np.mean(bests):.1f}±{np.std(bests, ddof=1) if len(bests)>1 else 0:.1f}',
            'Best−AUC': f'{np.mean(bests)-np.mean(aucs):.1f}',
            'n': len(runs),
        })
    return pd.DataFrame(rows_out).set_index('Method')

print(f'=== YELP α=0.1 — Extended Metrics ===')
display(extended_table('yelp', 0.1, 'accuracy'))

print(f'\n=== YELP α=0.01 — Extended Metrics ===')
display(extended_table('yelp', 0.01, 'accuracy'))

In [ ]:
# ── LaTeX main table snippet ──────────────────────────────────────────────────
def latex_row(method, yelp_alphas, gsm8k_alphas, df_):
    label = METHOD_LABELS.get(method, method)
    if method == 'hetlora_m':
        label = f'\\textbf{{{label}}}'
    parts = [label]

    # Yelp
    for a in yelp_alphas:
        g = df_[(df_.dataset=='yelp') & (df_.method==method) & (df_.alpha==a)]
        if len(g)==0:
            parts.append('—')
        else:
            parts.append(f'${g.auc.mean():.1f} \\pm {g.auc.std(ddof=1) if len(g)>1 else 0:.1f}$')

    # GSM8K
    for a in gsm8k_alphas:
        g = df_[(df_.dataset=='gsm8k') & (df_.method==method) & (df_.alpha==a)]
        if len(g)==0:
            parts.append('—')
        else:
            parts.append(f'${g.auc.mean():.1f} \\pm {g.auc.std(ddof=1) if len(g)>1 else 0:.1f}$')

    # Alpaca
    g = df_[(df_.dataset=='alpaca') & (df_.method==method)]
    if len(g)==0:
        parts.append('—')
    else:
        parts.append(f'${g.auc.mean():.3f} \\pm {g.auc.std(ddof=1) if len(g)>1 else 0:.3f}$')

    return ' & '.join(parts) + ' \\\\'

YELP_ALPHAS  = [0.5, 0.1, 0.01]
GSM8K_ALPHAS = [0.5, 0.1, 0.01]

print('%% LaTeX main table rows — paste into submission_v4.tex')
print()
for method in METHODS_ORDER:
    print(latex_row(method, YELP_ALPHAS, GSM8K_ALPHAS, df))